# M8.2 · Bitirme projesi: U-Net ile koşullu difüzyon ve segmentasyon

Kursun görüntü tarafını tek bir araştırma prototipinde birleştiriyorsun: zaman ve sınıf embedding'li, GroupNorm'lu artık bloklar, düşük çözünürlükte öz-attention ve atlama bağlantılarıyla küçük bir U-Net; sınıflandırıcısız yönlendirmeli koşullu difüzyon eğitimi, EMA ağırlıkları, DDIM ile hızlı örnekleme ve sınıflandırıcı öznitelikleriyle FID/sınıf doğruluğu değerlendirmesi. Yanında aynı mimari ailesinin ikinci kullanımı: sınıf dengesizliği olan küçük "lezyon" segmentasyonunda BCE, Dice ve karışık kayıpların hassasiyet–duyarlılık ödünleşimi. Araştırma pratiği: RePaint tarzı difüzyonla tamamlama (inpainting) ve neden kolay olmadığı, eşleştirilmiş bootstrap ile iki modeli dürüstçe karşılaştırma.

Etkileşimli sürüm: [https://m-gonul.github.io/pytorch-lab/m8-2-capstone.html](https://m-gonul.github.io/pytorch-lab/m8-2-capstone.html)

**Nasıl çalışılır:** her aşamada önce `%%writefile` hücresindeki kodu düzenleyip çalıştır (dosyaya yazar), sonra altındaki **Kontrol** hücresini çalıştır. Kontroller sitedekiyle birebir aynı. GPU gerekmez; Faz 7 modülleri hariç.

In [ ]:
#@title Kurulum: kontrol kütüphanesi (çalıştır, düzenleme)
%%writefile labcheck.py
"""labcheck — pytorch-lab'in ortak kontrol çekirdeği.

Aynı dosya üç yerde birebir çalışır:
  * tarayıcıda (Pyodide, Web Worker içinde),
  * yerel runner'da (uv run runner/torchlab.py),
  * CI'da ve Colab notebook'larında.

Protokol: öğrenci kodu `exec` ile ayrı bir namespace'te çalıştırılır, ardından
aşamanın kontrol dosyası (`check(ns, R)` fonksiyonu tanımlar) aynı namespace
üzerinde çalışır. Sonuç JSON'a çevrilebilir bir dict olarak döner.
"""
from __future__ import annotations

import ast
import contextlib
import io
import json
import math
import time
import traceback

__all__ = ["Report", "run_stage", "close", "to_list", "no_loops", "no_imports",
           "uses_any", "need", "ensure", "bench", "fmt_time", "loop_depth", "max_loops"]


# --------------------------------------------------------------------------- yardımcılar
def to_list(x):
    """Tensor / ndarray / liste / skaleri iç içe Python listesine (float) çevirir."""
    if hasattr(x, "detach"):  # torch.Tensor
        x = x.detach().cpu()
        x = x.tolist()
    elif hasattr(x, "tolist"):  # numpy
        x = x.tolist()
    if isinstance(x, (list, tuple)):
        return [to_list(v) for v in x]
    if isinstance(x, bool):
        return float(x)
    if isinstance(x, (int, float)):
        return float(x)
    return x


def _shape(x):
    if isinstance(x, list):
        return (len(x),) + (_shape(x[0]) if x else ())
    return ()


def _flat(x):
    if isinstance(x, list):
        for v in x:
            yield from _flat(v)
    else:
        yield x


def close(got, want, rtol=1e-5, atol=1e-6, what="sonuç"):
    """`got` ile `want` aynı shape'te ve sayısal olarak yakın mı? Değilse AssertionError."""
    g, w = to_list(got), to_list(want)
    if _shape(g) != _shape(w):
        raise AssertionError(f"{what}: shape {_shape(g)} bekleniyordu {_shape(w)}")
    worst = 0.0
    for a, b in zip(_flat(g), _flat(w)):
        if not isinstance(a, float):
            raise AssertionError(f"{what}: sayı olmayan değer {a!r}")
        if math.isnan(a) and math.isnan(b):
            continue
        err = abs(a - b)
        if err > atol + rtol * abs(b):
            raise AssertionError(f"{what}: {a:.6g} bulundu, {b:.6g} bekleniyordu")
        worst = max(worst, err)
    return worst


def _tree(src):
    try:
        return ast.parse(src)
    except SyntaxError:
        return None


def no_loops(src, allow_comprehension=False):
    """Kaynakta `for`/`while` döngüsü (ve istenirse comprehension) var mı? Varsa AssertionError."""
    t = _tree(src)
    if t is None:
        return
    banned = (ast.For, ast.While, ast.AsyncFor)
    if not allow_comprehension:
        banned += (ast.ListComp, ast.SetComp, ast.DictComp, ast.GeneratorExp)
    for node in ast.walk(t):
        if isinstance(node, banned):
            kind = "döngü" if isinstance(node, (ast.For, ast.While, ast.AsyncFor)) else "comprehension"
            raise AssertionError(f"satır {node.lineno}: bu aşamada {kind} yasak — işlemi vektörel yaz")


def _func(tree, name):
    for node in ast.walk(tree):
        if isinstance(node, (ast.FunctionDef, ast.AsyncFunctionDef)) and node.name == name:
            return node
    return None


def loop_depth(src, func=None):
    """Kaynakta (ya da yalnız `func` fonksiyonunda) iç içe for/while derinliği. Comprehension'lar da sayılır."""
    t = _tree(src)
    if t is None:
        return 0
    if func is not None:
        t = _func(t, func)
        if t is None:
            raise AssertionError(f"`{func}` fonksiyonu bulunamadı")
    loops = (ast.For, ast.While, ast.AsyncFor, ast.ListComp, ast.SetComp, ast.DictComp, ast.GeneratorExp)

    def depth(node):
        own = 1 if isinstance(node, loops) else 0
        if isinstance(node, (ast.ListComp, ast.SetComp, ast.DictComp, ast.GeneratorExp)):
            own = len(node.generators)
        return own + max((depth(c) for c in ast.iter_child_nodes(node)), default=0)
    return depth(t)


def max_loops(src, n, func=None, why=""):
    """Döngü derinliği en fazla n mi? (ör. eğitim adımları için tek döngü serbest, örnekler üzerinde döngü yasak)"""
    d = loop_depth(src, func)
    if d > n:
        where = f"`{func}` içinde " if func else ""
        raise AssertionError(f"{where}{d} katmanlı döngü var; en fazla {n} olmalı{(' — ' + why) if why else ''}")


def no_imports(src, names):
    """Belirtilen modüllerin import edilmediğini doğrular (ör. b aşamasında numpy yok)."""
    t = _tree(src)
    if t is None:
        return
    for node in ast.walk(t):
        mods = []
        if isinstance(node, ast.Import):
            mods = [a.name.split(".")[0] for a in node.names]
        elif isinstance(node, ast.ImportFrom) and node.module:
            mods = [node.module.split(".")[0]]
        for m in mods:
            if m in names:
                raise AssertionError(f"satır {node.lineno}: bu aşamada `{m}` kullanılmamalı")


def uses_any(src, words):
    """Kaynak, verilen isimlerden en az birini (attribute ya da isim olarak) kullanıyor mu?"""
    t = _tree(src)
    if t is None:
        return False
    found = set()
    for node in ast.walk(t):
        if isinstance(node, ast.Attribute):
            found.add(node.attr)
        elif isinstance(node, ast.Name):
            found.add(node.id)
    return any(w in found for w in words)


def ensure(cond, msg):
    """`cond` yanlışsa `msg` ile AssertionError."""
    if not cond:
        raise AssertionError(msg)


def need(ns, name, kind="fonksiyon"):
    if name not in ns:
        raise AssertionError(f"`{name}` adlı {kind} tanımlanmamış")
    return ns[name]


def fmt_time(s):
    if s < 1e-6:
        return f"{s * 1e9:.0f} ns"
    if s < 1e-3:
        return f"{s * 1e6:.1f} µs"
    if s < 1:
        return f"{s * 1e3:.2f} ms"
    return f"{s:.2f} s"


def bench(fn, repeat=5, number=None, budget=0.25):
    """`fn()`'in medyan süresi (saniye). number verilmezse ~budget saniyeye göre ayarlanır."""
    fn()  # ısınma (lazy init, torch.compile vb.)
    if number is None:
        t0 = time.perf_counter()
        fn()
        one = max(time.perf_counter() - t0, 1e-7)
        number = max(1, min(10_000, int(budget / one / repeat)))
    times = []
    for _ in range(repeat):
        t0 = time.perf_counter()
        for _ in range(number):
            fn()
        times.append((time.perf_counter() - t0) / number)
    times.sort()
    return times[len(times) // 2]


# --------------------------------------------------------------------------- rapor
class Report:
    def __init__(self):
        self.items = []    # {"name", "ok", "msg"}
        self.tables = []   # {"title", "cols", "rows", "note"}
        self.notes = []

    def test(self, name, fn=None):
        """`R.test("ad", lambda: ...)` ya da dekoratör olarak `@R.test("ad")`."""
        if fn is None:
            return lambda f: (self.test(name, f), f)[1]
        try:
            res = fn()
            msg = res if isinstance(res, str) else ""
            self.items.append({"name": name, "ok": True, "msg": msg})
        except AssertionError as e:
            self.items.append({"name": name, "ok": False, "msg": str(e) or "koşul sağlanmadı"})
        except Exception as e:  # öğrenci kodu patladı
            tb = traceback.extract_tb(e.__traceback__)
            where = next((f"satır {f.lineno}: " for f in reversed(tb) if f.filename == "<öğrenci>"), "")
            self.items.append({"name": name, "ok": False, "msg": f"{where}{type(e).__name__}: {e}"})
        return fn

    def table(self, title, cols, rows, note=""):
        self.tables.append({"title": title, "cols": list(cols), "rows": [list(map(str, r)) for r in rows], "note": note})

    def note(self, text):
        self.notes.append(str(text))

    @property
    def ok(self):
        return bool(self.items) and all(i["ok"] for i in self.items)


def _short_tb(e):
    tb = traceback.extract_tb(e.__traceback__)
    frames = [f for f in tb if f.filename == "<öğrenci>"]
    loc = f"satır {frames[-1].lineno}: " if frames else ""
    return f"{loc}{type(e).__name__}: {e}"


def run_stage(code, check_code=None, max_out=20_000):
    """Öğrenci kodunu çalıştırır, istenirse kontrolleri uygular. JSON'a çevrilebilir dict döner."""
    out = io.StringIO()
    ns = {"__name__": "__main__", "__source__": code}
    t0 = time.perf_counter()
    error = None
    with contextlib.redirect_stdout(out), contextlib.redirect_stderr(out):
        try:
            exec(compile(code, "<öğrenci>", "exec"), ns)
        except SyntaxError as e:
            error = f"satır {e.lineno}: SyntaxError: {e.msg}"
        except Exception as e:
            error = _short_tb(e)
    elapsed = time.perf_counter() - t0
    R = Report()
    if check_code is not None and error is None:
        cns = {"__name__": "labcheck_stage"}
        with contextlib.redirect_stdout(out):
            try:
                exec(compile(check_code, "<kontrol>", "exec"), cns)
                cns["check"](ns, R)
            except Exception as e:  # kontrol dosyası hatası: kullanıcıya göster
                R.items.append({"name": "kontrol çalıştırılamadı", "ok": False,
                                "msg": f"{type(e).__name__}: {e}"})
    text = out.getvalue()
    if len(text) > max_out:
        text = text[:max_out] + f"\n… (çıktı {len(text) - max_out} karakter kısaltıldı)"
    return {
        "stdout": text,
        "error": error,
        "elapsed": elapsed,
        "checked": check_code is not None,
        "ok": error is None and (check_code is None or R.ok),
        "items": R.items,
        "tables": R.tables,
        "notes": R.notes,
    }


def run_stage_json(code, check_code=None):
    return json.dumps(run_stage(code, check_code), ensure_ascii=False)


def run_file(path, check_code):
    """Colab/Jupyter yardımcısı: `%%writefile` ile yazılmış aşama dosyasını kontrol eder ve özetler."""
    with open(path, encoding="utf-8") as f:
        res = run_stage(f.read(), check_code)
    if res["stdout"].strip():
        print(res["stdout"].rstrip())
    if res["error"]:
        print("✗ hata:", res["error"])
    for it in res["items"]:
        print(("✓ " if it["ok"] else "✗ ") + it["name"] + (f" — {it['msg']}" if it["msg"] else ""))
    for t in res["tables"]:
        print(f"\n{t['title']}")
        w = [max(len(str(c)), *(len(r[i]) for r in t["rows"])) for i, c in enumerate(t["cols"])]
        print("  ".join(str(c).ljust(w[i]) for i, c in enumerate(t["cols"])))
        for r in t["rows"]:
            print("  ".join(v.ljust(w[i]) for i, v in enumerate(r)))
        if t["note"]:
            print(t["note"])
    print("\nSONUÇ:", "geçti ✓" if res["ok"] else "henüz değil ✗")
    return res["ok"]

In [ ]:
#@title Kurulum: bu modülün kontrolleri (çalıştır, düzenleme)
%%writefile checks_m8_2_capstone.py
CHECKS = {
 "b": "import math\nfrom labcheck import close, no_imports, need, ensure\n\n\ndef check(ns, R_):\n    src = ns[\"__source__\"]\n    R_.test(\"numpy kullanılmıyor\", lambda: no_imports(src, {\"numpy\", \"torch\"}))\n\n    def t():\n        e = [0.0]\n        for _ in range(1000):\n            e = need(ns, \"ema_update\")(e, [1.0], 0.999)\n        close(e[0], 1 - 0.999 ** 1000, what=\"EMA: sabit hedefe yaklaşma\")\n        emb = need(ns, \"timestep_embedding\")(50, 8)\n        close(emb[1], math.sin(5), what=\"ω₁ = 0.1\")\n        close(emb[4:], [math.cos(50 * 10000 ** (-i / 4)) for i in range(4)], what=\"cos yarısı\")\n        ts = need(ns, \"ddim_timesteps\")(200, 25)\n        ensure(ts[0] == 199 and ts[-1] == 0 and len(ts) == 25 and all(a > b for a, b in zip(ts, ts[1:])), \"azalan alt dizi\")\n        ensure(need(ns, \"nfe\")(25, True, False) == 50 and ns[\"nfe\"](25, True, True) == 25 and ns[\"nfe\"](200, False, False) == 200, \"NFE\")\n        close(need(ns, \"dice\")([1] * 30 + [0] * 20, [1] * 25 + [0] * 5 + [1] * 15 + [0] * 5), 2 * 25 / 70, what=\"Dice\")\n        ensure(ns[\"dice\"]([0, 0], [0, 0]) == 1.0, \"boş-boş\")\n        rows = []\n        for d in (0.9, 0.99, 0.995, 0.999, 0.9999):\n            rows.append([d, f\"{1 / (1 - d):.0f}\", f\"{d ** 1000:.3f}\"])\n        R_.table(\"EMA bozunması\", [\"β\", \"etkin pencere (adım)\", \"1000 adım önceki ağırlığın payı\"], rows,\n                 note=\"Kısa eğitimde (≈ 1000 adım) 0.9999 neredeyse başlangıç ağırlıklarında kalır — EMA'yı 'ısıtmak' (β'yı adım sayısıyla artırmak, ör. min(β, (1 + n)/(10 + n))) ya da eğitim süresine uygun β seçmek gerekir.\")\n    R_.test(\"bütçe hesapları\", t)\n",
 "c": "import numpy as np\nfrom labcheck import close, max_loops, need, ensure\n\n\ndef check(ns, R_):\n    src = ns[\"__source__\"]\n    R_.test(\"döngüsüz\", lambda: [max_loops(src, 0, func=f) for f in (\"group_norm\", \"conv3x3\", \"resblock\", \"attn2d\")])\n\n    def t():\n        rng = np.random.default_rng(0)\n        C, H = 8, 6\n        x = rng.normal(2, 3, (C, H, H))\n        g = need(ns, \"group_norm\")(x, 4)\n        gg = g.reshape(4, -1)\n        close((gg.mean(1), gg.std(1)), (np.zeros(4), np.ones(4)), atol=1e-4, what=\"GroupNorm grupları\")\n        close(ns[\"group_norm\"](x * 5 + 1, 4), g, atol=1e-4, what=\"ölçek ve kaymadan bağımsız\")\n        W, b = rng.normal(size=(5, C, 3, 3)), rng.normal(size=5)\n        y = need(ns, \"conv3x3\")(x, W, b)\n        close(y[2, 3, 3], np.sum(W[2] * x[:, 2:5, 2:5]) + b[2], what=\"konvolüsyon\")\n        P = {\"G\": 4, \"W1\": rng.normal(size=(C, C, 3, 3)) * 0.1, \"b1\": np.zeros(C), \"Wt\": rng.normal(size=(C, 16)), \"bt\": np.zeros(C), \"W2\": np.zeros((C, C, 3, 3)), \"b2\": np.zeros(C)}\n        te = rng.normal(size=16)\n        close(need(ns, \"resblock\")(x, te, P), x, what=\"sıfır başlatılmış ikinci konv. → blok özdeşlik\")\n        P[\"W2\"] = rng.normal(size=(C, C, 3, 3)) * 0.1\n        r1, r2 = ns[\"resblock\"](x, te, P), ns[\"resblock\"](x, te + 1.0, P)\n        ensure(np.abs(r1 - r2).max() > 1e-3, \"zaman embedding'i çıktıyı değiştirir\")\n        A = {\"G\": 4, \"Wq\": rng.normal(size=(C, C)), \"Wk\": rng.normal(size=(C, C)), \"Wv\": rng.normal(size=(C, C)), \"Wo\": np.zeros((C, C))}\n        close(need(ns, \"attn2d\")(x, A), x, what=\"sıfır W_o → özdeşlik\")\n        A[\"Wo\"] = rng.normal(size=(C, C)) * 0.1\n        xp = x.reshape(C, -1)[:, rng.permutation(H * H)].reshape(C, H, H)\n        ensure(ns[\"attn2d\"](x, A).shape == x.shape, \"şekil\")\n        R_.note(\"Difüzyon U-Net'lerinde her artık bloğun ve attention bloğunun çıkış izdüşümü sıfırla başlatılır: ağ eğitime özdeşlik + giriş/çıkış konvolüsyonlarından oluşan sığ bir ağ olarak başlar. GroupNorm batch'ten bağımsızdır (küçük batch ve EMA ile güvenli) ve girdi ölçeğini unutur — zaman embedding'i bu yüzden normalizasyondan sonra eklenir ki 'ne kadar gürültü var' bilgisi silinmesin.\")\n    R_.test(\"U-Net blokları\", t)\n",
 "d": "import time\nimport numpy as np\nimport torch\nimport torch.nn as nn\nimport torch.nn.functional as F\nfrom labcheck import need, ensure\n\n\ndef _fid(a, b):\n    m1, m2 = a.mean(0), b.mean(0)\n    S1, S2 = np.cov(a, rowvar=False), np.cov(b, rowvar=False)\n    w, V = np.linalg.eigh(S1)\n    r = (V * np.sqrt(np.clip(w, 0, None))) @ V.T\n    return float(((m1 - m2) ** 2).sum() + np.trace(S1 + S2) - 2 * np.sqrt(np.clip(np.linalg.eigvalsh(r @ S2 @ r), 0, None)).sum())\n\n\ndef check(ns, R_):\n    def t_arch():\n        torch.manual_seed(0)\n        m = need(ns, \"UNet\")(16, 10)\n        x = torch.randn(3, 1, 16, 16)\n        out = m(x, torch.tensor([0, 50, 199]), torch.tensor([1, 2, 10]))\n        ensure(tuple(out.shape) == (3, 1, 16, 16), \"çıkış (B, 1, 16, 16)\")\n        rb = need(ns, \"ResBlock\")(8, 8, 32)\n        z = torch.randn(2, 8, 4, 4)\n        ensure(torch.allclose(rb(z, torch.randn(2, 32)), z), \"sıfır başlatılmış blok özdeşlik\")\n        at = need(ns, \"Attn\")(8)\n        ensure(torch.allclose(at(z), z), \"attention başlangıçta özdeşlik\")\n        ensure(m.cemb.num_embeddings == 11, \"10 sınıf + boş token\")\n    R_.test(\"mimari\", t_arch)\n\n    def t_train():\n        X, y = need(ns, \"digits16\")()\n        torch.manual_seed(0)\n        clf = nn.Sequential(nn.Conv2d(1, 32, 3, padding=1), nn.ReLU(), nn.MaxPool2d(2), nn.Conv2d(32, 64, 3, padding=1), nn.ReLU(), nn.MaxPool2d(2),\n                            nn.Flatten(), nn.Linear(1024, 64), nn.ReLU(), nn.Linear(64, 10))\n        o = torch.optim.Adam(clf.parameters(), 1e-3)\n        for _ in range(15):\n            for i in range(0, len(X), 64):\n                p = torch.randperm(len(X))[:64]\n                loss = F.cross_entropy(clf(X[p]), y[p])\n                o.zero_grad()\n                loss.backward()\n                o.step()\n        clf.eval()\n        feat = lambda z: clf[:-1](z).detach().numpy()\n        fr = feat(X)\n        base = _fid(feat(X[::2]), feat(X[1::2]))\n        torch.manual_seed(0)\n        t0 = time.perf_counter()\n        m, ema = need(ns, \"train\")(ns[\"UNet\"](16, 10), 1200, 0, 0.99)\n        tt = time.perf_counter() - t0\n        labels = torch.arange(10).repeat(50)\n        rows, R = [], {}\n        for name, net in ((\"ham ağırlıklar\", m), (\"EMA (0.99)\", ema)):\n            for s, steps in ((1.0, 25), (2.0, 25), (1.0, 10)):\n                t1 = time.perf_counter()\n                xs = need(ns, \"sample\")(net, labels, s, steps, 1)\n                ts = time.perf_counter() - t1\n                with torch.no_grad():\n                    pr = clf(xs).softmax(1)\n                acc = (pr.argmax(1) == labels).float().mean().item()\n                R[(name, s, steps)] = (acc, _fid(fr, feat(xs)))\n                rows.append([name, s, steps, f\"{ts:.1f} sn\", f\"{acc:.3f}\", f\"{R[(name, s, steps)][1]:.1f}\"])\n        e1, r1 = R[(\"EMA (0.99)\", 1.0, 25)], R[(\"ham ağırlıklar\", 1.0, 25)]\n        ensure(e1[1] < r1[1] and e1[0] > 0.75, f\"EMA daha iyi FID ve makul sınıf doğruluğu {R}\")\n        ensure(R[(\"EMA (0.99)\", 2.0, 25)][0] > e1[0], \"yönlendirme sınıf doğruluğunu artırır\")\n        R_.table(f\"Koşullu U-Net difüzyonu, 16 × 16 rakamlar, 1200 adım ({tt:.0f} sn) — FID taban çizgisi (gerçek–gerçek) {base:.1f}\", [\"ağırlıklar\", \"s\", \"DDIM adımı\", \"örnekleme (500)\", \"sınıf doğruluğu\", \"FID\"], rows,\n                 note=\"Aynı eğitim koşusunda EMA ağırlıkları FID'i belirgin düşürür: difüzyon örneklemesi ağı onlarca kez zincirlediği için ağırlıklardaki SGD gürültüsü örneklerde birikir. s = 2 sınıf doğruluğunu artırır ama FID'i iyileştirmeyebilir (çeşitlilik ↓, aşırı doyma). DDIM'de 10 adım bu küçük problemde 25'e yakın kalite verir.\")\n    R_.test(\"eğitim ve değerlendirme\", t_train)\n",
 "e": "import math\nimport time\nimport torch\nimport torch.nn as nn\nimport torch.nn.functional as F\nfrom labcheck import close, need, ensure\n\n\nclass _Net(nn.Module):\n    def __init__(self, c=16, C=10):\n        super().__init__()\n        self.C = C\n        self.emb = nn.Embedding(C + 1, c)\n        self.body = nn.Sequential(nn.Conv2d(1 + c, c, 3, padding=1), nn.SiLU(), nn.Conv2d(c, c, 3, padding=1), nn.SiLU(), nn.Conv2d(c, 1, 3, padding=1))\n\n    def forward(self, x, t, y):\n        e = self.emb(y)[:, :, None, None].expand(-1, -1, x.shape[2], x.shape[3]) + (t.float() / 200)[:, None, None, None]\n        return self.body(torch.cat([x, e], 1))\n\n\ndef check(ns, R_):\n    def t():\n        torch.manual_seed(0)\n        net = _Net().eval()\n        x = torch.randn(8, 1, 16, 16)\n        tt = torch.full((8,), 37)\n        y = torch.arange(8)\n        with torch.no_grad():\n            a = need(ns, \"cfg_two_calls\")(net, x, tt, y, 2.5)\n            b = need(ns, \"cfg_batched\")(net, x, tt, y, 2.5)\n        close(a.numpy(), b.numpy(), atol=1e-5, what=\"toplu CFG = iki çağrı\")\n        labels = torch.arange(10).repeat(20)\n        d1, d2 = need(ns, \"ddim_sample\")(net, labels, 2.0, 25, 1), ns[\"ddim_sample\"](net, labels, 2.0, 25, 1)\n        ensure(torch.equal(d1, d2), \"DDIM deterministik (aynı tohum)\")\n        p1, p2 = need(ns, \"ddpm_sample\")(net, labels, 2.0, 1), ns[\"ddpm_sample\"](net, labels, 2.0, 2)\n        ensure(tuple(p1.shape) == (200, 1, 16, 16) and not torch.equal(p1, p2), \"DDPM stokastik\")\n        rows = []\n        with torch.no_grad():\n            for name, f in ((\"iki çağrı\", ns[\"cfg_two_calls\"]), (\"toplu\", ns[\"cfg_batched\"])):\n                f(net, x, tt, y, 2.0)\n                t0 = time.perf_counter()\n                for _ in range(20):\n                    f(net, torch.randn(64, 1, 16, 16), torch.full((64,), 10), torch.arange(64) % 10, 2.0)\n                rows.append([f\"CFG {name} (batch 64, 20 kez)\", f\"{time.perf_counter() - t0:.3f} sn\"])\n        for name, f in ((\"DDPM 200 adım\", lambda: ns[\"ddpm_sample\"](net, labels, 2.0, 0)), (\"DDIM 25 adım\", lambda: ns[\"ddim_sample\"](net, labels, 2.0, 25, 0)), (\"DDIM 10 adım\", lambda: ns[\"ddim_sample\"](net, labels, 2.0, 10, 0))):\n            t0 = time.perf_counter()\n            f()\n            rows.append([f\"{name} (200 örnek)\", f\"{time.perf_counter() - t0:.3f} sn\"])\n        R_.table(\"Çıkarım maliyeti (CPU, küçük ağ)\", [\"işlem\", \"süre\"], rows,\n                 note=\"Koşullu ve koşulsuz tahmini tek batch'te hesaplamak çağrı ve çekirdek başlatma yükünü yarıya indirir (GPU'da büyük batch verimli olduğu için kazanç daha da büyüktür). Örnekleme süresi adım sayısıyla doğrusal: d aşamasında DDIM-10'un kalitesi DDIM-25'e yakındı, DDPM-200'ün 20 katı daha ucuz. Damıtma (progressive distillation, consistency models) bu sayıyı 1–4 adıma indirir.\")\n    R_.test(\"çıkarım mühendisliği\", t)\n",
 "x1": "import time\nimport torch\nfrom labcheck import close, need, ensure\n\n\ndef check(ns, R_):\n    def t():\n        lo = torch.tensor([[[10.0, -10.0], [-10.0, 10.0]]])\n        close(need(ns, \"dice_loss\")(lo, torch.tensor([[[1.0, 0.0], [0.0, 1.0]]])).item(), 0.0, atol=1e-3, what=\"mükemmel tahmin\")\n        close(need(ns, \"seg_metrics\")(torch.tensor([1.0, 1, 0, 0]), torch.tensor([1.0, 0, 1, 0])), (0.5, 0.5, 0.5), what=\"metrikler\")\n        Xt, Yt = ns[\"lesions\"](300, 1)\n        rows, R = [], {}\n        t0 = time.perf_counter()\n        for loss in (\"bce\", \"wbce\", \"dice\", \"bce+dice\"):\n            m = need(ns, \"train_seg\")(loss, 300, 0)\n            with torch.no_grad():\n                P = (m(Xt) > 0).float()\n            R[loss] = ns[\"seg_metrics\"](P, Yt)\n            rows.append([loss.upper(), f\"{R[loss][0]:.3f}\", f\"{R[loss][1]:.3f}\", f\"{R[loss][2]:.3f}\"])\n        ensure(R[\"wbce\"][1] > R[\"bce\"][1] and R[\"wbce\"][2] < R[\"bce\"][2], \"ağırlıklı BCE: duyarlılık ↑, kesinlik ↓\")\n        ensure(R[\"dice\"][1] > R[\"bce\"][1] and max(R[\"dice\"][0], R[\"bce+dice\"][0]) > R[\"bce\"][0] - 0.01, \"Dice duyarlılığı artırır\")\n        R_.table(f\"Küçük lezyonlar (ön plan ≈ %{100 * float(Yt.mean()):.0f}), 300 adım × 4 kayıp ({time.perf_counter() - t0:.0f} sn)\", [\"kayıp\", \"Dice\", \"duyarlılık (recall)\", \"kesinlik (precision)\"], rows,\n                 note=\"BCE piksel başına ortalandığı için baskın arka planı doğru bilmek yeter: kenar ve küçük lezyon pikselleri kaçırılır (yüksek kesinlik, düşük duyarlılık). Ön plan ağırlığı duyarlılığı yükseltip yanlış pozitif artırır. Dice kaybı ön plan örtüşmesini doğrudan optimize eder; BCE + Dice (nnU-Net varsayılanı) kalibre olasılık ve iyi örtüşmeyi birleştirir. Hangisinin 'iyi' olduğu klinik maliyete bağlıdır: lezyon kaçırmak mı daha pahalı, yanlış alarm mı?\")\n    R_.test(\"segmentasyon kayıpları\", t)\n",
 "x2": "import numpy as np\nfrom labcheck import need, ensure\n\n\ndef check(ns, R_):\n    def t():\n        rng = np.random.default_rng(0)\n        rows, wrong = [], {}\n        for U in (1, 5, 20):\n            x = need(ns, \"repaint\")(-0.6, 1500, rng, U)\n            ensure(np.allclose(x[:, 0], -0.6), \"bilinen koordinat korunur\")\n            wrong[U] = float((x[:, 1] > 0).mean())\n            rows.append([U, 200 * U, f\"{wrong[U]:.3f}\"])\n        p_true = 1 / (1 + np.exp(1.2 ** 2 / (2 * 0.09)))\n        rows.append([\"gerçek koşullu\", \"—\", f\"{p_true:.4f}\"])\n        ensure(wrong[1] > 0.2 and wrong[20] < wrong[1], f\"naif yerleştirme başarısız, yeniden örnekleme iyileştirir {wrong}\")\n        R_.table(\"x = −0.6 bilinirken y'nin yanlış moda (y > 0) düşme oranı (kâhin ε!)\", [\"yeniden örnekleme U\", \"ağ değerlendirmesi\", \"yanlış mod oranı\"], rows,\n                 note=\"Model kusursuz olsa bile naif 'bilineni yerleştir' yöntemi koşullu dağılımdan örneklemez: erken (gürültülü) adımlarda bilinen koordinatın gürültülü sürümü bilinmeyen koordinata neredeyse hiç bilgi taşımaz ve y kendi başına bir moda karar verir; sonradan düzeltilemez. RePaint'in yeniden örneklemesi (gürültüle–tekrar çöz) bilgiyi ileri geri yayarak yardımcı olur ama hesap pahalıdır ve hâlâ kesin değildir. Kesin çözüm için posteriora göre yönlendirme (DPS, ΠGDM) ya da koşullu eğitim gerekir — tıbbi görüntü tamamlamada bu fark klinik olarak önemlidir.\")\n    R_.test(\"difüzyonla tamamlama\", t)\n",
 "x3": "import numpy as np\nfrom labcheck import close, max_loops, need, ensure\n\n\ndef check(ns, R_):\n    src = ns[\"__source__\"]\n    R_.test(\"döngüsüz\", lambda: [max_loops(src, 0, func=f) for f in (\"paired_bootstrap\", \"unpaired_bootstrap\")])\n\n    def t():\n        rng = np.random.default_rng(0)\n        d, lo, hi = need(ns, \"paired_bootstrap\")(np.ones(10) * 2, np.ones(10), 200, rng)\n        close((d, lo, hi), (1.0, 1.0, 1.0), what=\"sabit fark\")\n        cover, widths_p, widths_u, sig_p, sig_u = 0, [], [], 0, 0\n        for k in range(200):\n            hard = rng.random(300)\n            a = np.clip(0.9 - 0.6 * hard + 0.05 * rng.normal(size=300), 0, 1)\n            b = np.clip(a - 0.01 + 0.03 * rng.normal(size=300), 0, 1)\n            dp, lp, hp = ns[\"paired_bootstrap\"](a, b, 400, rng)\n            du, lu, hu = need(ns, \"unpaired_bootstrap\")(a, b, 400, rng)\n            cover += lp <= 0.01 <= hp\n            widths_p.append(hp - lp)\n            widths_u.append(hu - lu)\n            sig_p += lp > 0\n            sig_u += lu > 0\n        ensure(0.88 <= cover / 200 <= 0.99, f\"%95 aralığın kapsama oranı {cover / 200:.2f}\")\n        ensure(np.mean(widths_p) < 0.3 * np.mean(widths_u), \"eşleştirme aralığı daraltır\")\n        R_.table(\"200 sentetik 'deney': örnek başına skorlar, gerçek ortalama fark 0.01 (model A, B'den biraz iyi)\", [\"yöntem\", \"ort. %95 aralık genişliği\", \"farkı anlamlı bulma oranı\"], [[\"eşleştirilmiş bootstrap\", f\"{np.mean(widths_p):.4f}\", f\"{sig_p / 200:.2f}\"], [\"eşleştirilmemiş bootstrap\", f\"{np.mean(widths_u):.4f}\", f\"{sig_u / 200:.2f}\"]],\n                 note=f\"Örnek zorluğu iki modeli birlikte etkiler (zor görüntüde ikisi de kötü): bu ortak varyans eşleştirilmemiş karşılaştırmada gürültü olarak kalır, küçük ama gerçek bir farkı gizler. Aynı test örnekleri üzerinde farkları yeniden örneklemek bu varyansı yok eder. %95 aralığın kapsama oranı {cover / 200:.2f}. Raporlarken tohum varyansını da (birkaç eğitim koşusu) ekle — bootstrap yalnız test kümesi belirsizliğini ölçer.\")\n    R_.test(\"eşleştirilmiş bootstrap\", t)\n"
}

In [ ]:
import importlib, labcheck, checks_m8_2_capstone as _c
importlib.reload(labcheck); importlib.reload(_c)
CHECKS = _c.CHECKS
print("hazır")

## Giriş

Bu modül bir proje şablonudur: önceki fazlardan parçaları (konvolüsyon ve U-Net — M4.10, normalizasyon — M3.6, attention — M4.6, difüzyon ve yönlendirme — M5.3/M5.8, eğitim döngüsü ve profil — M8.1, değerlendirme — M5.7) tek bir çalışan sisteme bağlıyorsun. İki iz var: **üretim** (koşullu difüzyon) ve **analiz** (segmentasyon); ikisi de aynı U-Net ailesini kullanır.

**Difüzyon U-Net'i** (Ho ve ark. 2020'deki PixelCNN++ tabanlı mimarinin küçültülmüşü): giriş konvolüsyonu → çözünürlük başına artık bloklar (GroupNorm → SiLU → 3×3 konv.; zaman + sınıf embedding'i doğrusal izdüşümle kanal başına eklenir; ikinci konvolüsyon sıfır başlatılır) → adımlı konvolüsyonla küçültme → en düşük çözünürlükte öz-attention → yukarı örnekleme + atlama birleştirmesi → çıkış. Zaman sinüzoidal embedding'le (M4.14) kodlanır. Eğitim: ε-tahmini MSE, etiketin %10'u boş token (CFG), ağırlıkların **EMA**'sı ile örnekleme.

**Değerlendirme disiplini:** FID'i her zaman gerçek–gerçek taban çizgisiyle birlikte raporla (sonlu örneklem yanlılığı, M5.7 x2); sınıf koşullu modellerde sınıf doğruluğunu ayrı ölç; yönlendirme ölçeği kaliteyi ve çeşitliliği farklı yönlere iter. İki modeli karşılaştırırken tek bir sayıya değil, aynı test örnekleri üzerindeki eşleştirilmiş farkın güven aralığına bak.

**Segmentasyon izi:** ön plan piksellerinin ~%5 olduğu "lezyon" görüntüleri — tıbbi görüntülemenin tipik dengesizliği. Kayıp seçimi (BCE, ağırlıklı BCE, Dice, BCE + Dice) hassasiyet ile duyarlılık arasında seçim yapmaktır; nnU-Net'in varsayılanı Dice + CE'dir.

## 1 · Elle

EMA: $0.999^{1000}=e^{1000\ln0.999}\approx e^{-1}=0.368$ — "etkin pencere" $1/(1-\beta)=1000$ adım, bu kadar eski ağırlıklar hâlâ ~%37 katkıdadır. 0.995 → 200 adım: kısa eğitimde daha uygun (d aşaması 1200 adım).

CFG'de her adımda koşullu ve koşulsuz iki tahmin: 25 · 2 = 50 değerlendirme; toplu CFG (iki batch'i birleştirip tek çağrı) çağrı sayısını 25'e indirir ama hesap aynıdır (e aşaması).

Zaman embedding'i: $\omega_1=10000^{-0.25}=0.1$, $\sin(5)=-0.959$.

16 → 8 → 4: 4 × 4 = 16 token, skor matrisi 16² = 256 — attention'ı yalnız en düşük çözünürlükte kullanmanın nedeni (16 × 16'da 65 536).

Atlama: yukarı örneklenmiş 2c = 32 kanal + kodlayıcıdan c = 16 → 48.

Dice: 2 · 25 / (30 + 40) = 0.714 (IoU = 25 / 45 = 0.556).


Sitede bu aşama etkileşimli bir formdur. Burada değerleri aşağıdaki sözlüğe yazıp kontrol hücresini çalıştır.

**Eğitim ve çıkarım bütçesi**

- `ema`: EMA bozunması 0.999: 1000 adım önceki ağırlığın payı $0.999^{1000}$
- `win`: EMA bozunması 0.995'in etkin penceresi $1/(1-\beta)$ (adım)
- `nfe`: DDIM 25 adım + CFG (koşullu ve koşulsuz ayrı çağrılar): ağ değerlendirmesi sayısı
- `temb`: Zaman embedding'i d = 8: frekans $\omega_1=10000^{-1/4}$, t = 50 → $\sin(\omega_1t)$

**Mimari ve ölçüt**

- `att`: 16 × 16 girdi, iki 2× küçültme sonrası öz-attention: skor matrisi eleman sayısı (tek örnek, tek baş)
- `skip`: c = 16: en üst kod çözücü bloğunun girdi kanalı (2c'lik yukarı örneklenmiş + c'lik atlama)
- `dice`: Tahmin 30 piksel, gerçek 40, kesişim 25: Dice

In [ ]:
cevaplar = dict(
    ema=None,
    win=None,
    nfe=None,
    temb=None,
    att=None,
    skip=None,
    dice=None,
)

In [ ]:
#@title Kontrol (1)
_A = {"ema":[0.36769542477096373,0.000001],"win":[200,1e-9],"nfe":[50,0],"temb":[-0.9589242746631385,0.000001],"att":[256,0],"skip":[48,0],"dice":[0.7142857142857143,0.000001]}
_ok = 0
for k, (v, tol) in _A.items():
    g = cevaplar.get(k)
    good = g is not None and abs(float(g) - v) <= max(tol, 1e-9)
    _ok += good
    print(("✓ " if good else ("· " if g is None else "✗ ")) + k)
print(f"\n{_ok}/{len(_A)} doğru")

## 2 · Döngü (saf Python)

Saf Python.

- `ema_update(ema, params, decay)` — listeler; yeni EMA listesi.
- `timestep_embedding(t, d)` — `[sin(t ω_0), …, sin(t ω_{d/2−1}), cos(t ω_0), …]`, $\omega_i=10000^{-i/(d/2)}$.
- `ddim_timesteps(T, steps)` — T − 1'den 0'a eşit aralıklı, yuvarlanmış tamsayı adımlar (azalan).
- `nfe(steps, cfg, batched)` — ağ çağrısı sayısı.
- `dice(pred, gt)` — ikili düz listeler; ikisi de boşsa 1.

In [ ]:
%%writefile stage_b.py
import math


def ema_update(ema, params, decay):
    ...


def timestep_embedding(t, d):
    ...


def ddim_timesteps(T, steps):
    ...


def nfe(steps, cfg, batched):
    ...


def dice(pred, gt):
    ...

In [ ]:
# Kontrol (2): kodunu çalıştırır ve sitedeki testleri uygular
labcheck.run_file("stage_b.py", CHECKS["b"])

## 3 · NumPy / pandas

NumPy, döngüsüz. Görüntü `(C, H, W)`, tek örnek.

- `group_norm(x, G, eps=1e-5)` — kanal grupları üzerinde normalize (γ = 1, β = 0).
- `conv3x3(x, W, b)` — "same" dolgu, `W (Co, Ci, 3, 3)`.
- `resblock(x, temb, P)` — $h=\mathrm{conv}_1(\mathrm{SiLU}(\mathrm{GN}(x)))+(W_tt_{emb})$ (kanal başına), $h=\mathrm{conv}_2(\mathrm{SiLU}(\mathrm{GN}(h)))$, çıktı $x+h$ (Ci = Co). `P` sözlüğü: `W1, b1, Wt, bt, W2, b2, G`.
- `attn2d(x, P)` — $x+W_o\,\mathrm{softmax}(QK^\top/\sqrt C)V$, Q, K, V GroupNorm'dan sonra 1×1 izdüşümler; `P`: `Wq, Wk, Wv, Wo, G`.

In [ ]:
%%writefile stage_c.py
import numpy as np
from numpy.lib.stride_tricks import sliding_window_view


def group_norm(x, G, eps=1e-5):
    ...


def conv3x3(x, W, b):
    ...


def resblock(x, temb, P):
    ...


def attn2d(x, P):
    ...

In [ ]:
# Kontrol (3): kodunu çalıştırır ve sitedeki testleri uygular
labcheck.run_file("stage_c.py", CHECKS["c"])

## 4 · PyTorch

PyTorch. Veri: 16 × 16'ya büyütülmüş rakamlar, [−1, 1]; T = 200, β 10⁻⁴'ten 0.05'e doğrusal. Verilenler: `digits16`, `temb`, `AB`.

- `ResBlock(ci, co, e)`, `Attn(c)`, `UNet(c=16, C=10)` — intro'daki mimari (GroupNorm(4, ·)); `forward(x, t, y)`; boş token indeksi C.
- `train(model, steps, seed, ema_decay=0.99, p_uncond=0.1)` → `(model, ema)` — AdamW 2e-3, batch 64; her adımdan sonra EMA güncellemesi.
- `sample(net, labels, s, steps, seed)` — CFG'li deterministik DDIM; her adımda $\hat x_0$ [−1, 1]'e kırpılır (dinamik olmayan eşikleme).

In [ ]:
%%writefile stage_d.py
import copy
import math
import torch
import torch.nn as nn
import torch.nn.functional as F
from sklearn.datasets import load_digits

T = 200
BETAS = torch.linspace(1e-4, 0.05, T)
AB = torch.cumprod(1 - BETAS, 0)


def digits16():
    X, y = load_digits(return_X_y=True)
    X = torch.tensor(X, dtype=torch.float32).view(-1, 1, 8, 8) / 16
    return F.interpolate(X, size=16, mode="bilinear", align_corners=False) * 2 - 1, torch.tensor(y)


def temb(t, d):
    f = torch.exp(-math.log(10000) * torch.arange(d // 2) / (d // 2))
    a = t[:, None].float() * f[None]
    return torch.cat([a.sin(), a.cos()], 1)


class ResBlock(nn.Module):
    def __init__(self, ci, co, e):
        super().__init__()
        ...

    def forward(self, x, e):
        ...


class Attn(nn.Module):
    def __init__(self, c):
        super().__init__()
        ...

    def forward(self, x):
        ...


class UNet(nn.Module):
    def __init__(self, c=16, C=10):
        super().__init__()
        ...

    def forward(self, x, t, y):
        ...


def train(model, steps, seed, ema_decay=0.99, p_uncond=0.1):
    ...


@torch.no_grad()
def sample(net, labels, s, steps, seed):
    ...

In [ ]:
# Kontrol (4): kodunu çalıştırır ve sitedeki testleri uygular
labcheck.run_file("stage_d.py", CHECKS["d"])

## 5 · Vektörize + hız

Çıkarım mühendisliği (eğitilmemiş bir U-Net üzerinde süre — süre ağırlıklardan bağımsızdır). Verilen: d'deki U-Net'in bir kopyası.

- `cfg_two_calls(net, x, t, y, s)` ve `cfg_batched(net, x, t, y, s)` — aynı $\hat\varepsilon$; toplu sürüm koşullu ve koşulsuz girdileri tek batch'te birleştirir.
- `ddpm_sample(net, labels, s, seed)` — 200 adımlı stokastik DDPM (σ² = β̃).
- `ddim_sample(net, labels, s, steps, seed)` — d'deki gibi.

In [ ]:
%%writefile stage_e.py
import math
import time
import torch
import torch.nn as nn
import torch.nn.functional as F

T = 200
BETAS = torch.linspace(1e-4, 0.05, T)
AB = torch.cumprod(1 - BETAS, 0)


def cfg_two_calls(net, x, t, y, s):
    ...


def cfg_batched(net, x, t, y, s):
    ...


@torch.no_grad()
def ddpm_sample(net, labels, s, seed):
    ...


@torch.no_grad()
def ddim_sample(net, labels, s, steps, seed):
    ...

In [ ]:
# Kontrol (5): kodunu çalıştırır ve sitedeki testleri uygular
labcheck.run_file("stage_e.py", CHECKS["e"])

## Ek alıştırmalar

Beş aşamanın dışındaki pratikler; her biri kendi kontrolüyle.

### E1 · Segmentasyon izi: küçük lezyonlarda BCE, Dice ve karışık kayıp (PyTorch)

PyTorch. Verilen: `lesions(n, seed)` (32 × 32 gürültülü doku üzerinde 0–3 küçük parlak disk; ön plan ≈ %5) ve küçük bir U-Net (`SegNet`).

- `dice_loss(logits, y, eps=1.0)` — yumuşak Dice, örnek başına, ortalama.
- `train_seg(loss, steps, seed)` — `loss ∈ {"bce", "wbce", "dice", "bce+dice"}` (wbce: pos_weight = 20); Adam 3e-3, batch 32; eğitilmiş model.
- `seg_metrics(P, Y)` → `(dice, recall, precision)` — tüm test pikselleri üzerinden.

In [ ]:
%%writefile extra_x1.py
import torch
import torch.nn as nn
import torch.nn.functional as F


def lesions(n, seed, S=32):
    g = torch.Generator().manual_seed(seed)
    yy, xx = torch.meshgrid(torch.arange(S).float(), torch.arange(S).float(), indexing="ij")
    X = F.avg_pool2d(0.3 * torch.randn(n, 1, S, S, generator=g), 3, 1, 1) * 3
    Y = torch.zeros(n, S, S)
    for i in range(n):
        for _ in range(int(torch.randint(0, 4, (1,), generator=g))):
            r = 1.5 + 3 * float(torch.rand(1, generator=g))
            cx, cy = (torch.rand(2, generator=g) * (S - 8) + 4).tolist()
            m = (xx - cx) ** 2 + (yy - cy) ** 2 <= r * r
            Y[i][m] = 1
            X[i, 0][m] += 0.8
    return X + 0.3 * torch.randn(X.shape, generator=g), Y


def _cbr(a, b):
    return nn.Sequential(nn.Conv2d(a, b, 3, padding=1), nn.BatchNorm2d(b), nn.ReLU())


class SegNet(nn.Module):
    def __init__(self, c=16):
        super().__init__()
        self.e1 = nn.Sequential(_cbr(1, c), _cbr(c, c))
        self.e2 = nn.Sequential(_cbr(c, 2 * c), _cbr(2 * c, 2 * c))
        self.up = nn.ConvTranspose2d(2 * c, c, 2, 2)
        self.d1 = nn.Sequential(_cbr(2 * c, c), _cbr(c, c))
        self.out = nn.Conv2d(c, 1, 1)

    def forward(self, x):
        a = self.e1(x)
        return self.out(self.d1(torch.cat([self.up(self.e2(F.max_pool2d(a, 2))), a], 1)))[:, 0]


def dice_loss(logits, y, eps=1.0):
    ...


def train_seg(loss, steps, seed):
    ...


def seg_metrics(P, Y):
    ...

In [ ]:
# Kontrol (E1)
labcheck.run_file("extra_x1.py", CHECKS["x1"])

### E2 · Difüzyonla tamamlama (RePaint) neden zor: kâhin modelle deney

NumPy. Veri: iki bileşenli 2B karışım $(-0.6,-2)$ ve $(0.6,2)$, σ = 0.3; kâhin ε verilir (M5.8). Görev: birinci koordinat $x=-0.6$ biliniyor, ikinciyi örnekle. Gerçek koşullu dağılım neredeyse tamamen $y\approx-2$'dir.

- `repaint(xk, n, rng, U)` — her ters adımda: bilinmeyen koordinat için $p_\theta(x_{t-1}|x_t)$'den örnekle (DDPM, σ² = β̃), bilinen koordinatı $q(x_{t-1}|x_0=xk)$'dan gürültülendirip yerleştir; `U > 1` ise (son tekrar hariç) bir adım ileri gürültüleyip ($x_t\sim N(\sqrt{\alpha_t}x_{t-1},\beta_t)$) aynı adımı tekrarla (RePaint yeniden örneklemesi). Son örnekler `(n, 2)`.

In [ ]:
%%writefile extra_x2.py
import numpy as np

MU = np.array([[-0.6, -2.0], [0.6, 2.0]])
SIG = 0.3
BETAS = np.linspace(1e-4, 0.05, 200)
AB = np.cumprod(1 - BETAS)


def eps(x, ab):
    v = ab * SIG ** 2 + 1 - ab
    m = np.sqrt(ab) * MU
    lw = -((x[:, None] - m[None]) ** 2).sum(-1) / (2 * v)
    r = np.exp(lw - lw.max(1, keepdims=True))
    r /= r.sum(1, keepdims=True)
    return np.sqrt(1 - ab) * (x - r @ m) / v


def repaint(xk, n, rng, U):
    ...

In [ ]:
# Kontrol (E2)
labcheck.run_file("extra_x2.py", CHECKS["x2"])

### E3 · İki modeli dürüstçe karşılaştırmak: eşleştirilmiş bootstrap

NumPy.

- `paired_bootstrap(a, b, B, rng, alpha=0.05)` — `a, b` aynı test örneklerindeki skorlar (ör. örnek başına Dice); farkların ortalaması ve yüzdelik bootstrap güven aralığı `(fark, alt, üst)`. Döngüsüz (B × n indeks matrisiyle).
- `unpaired_bootstrap(a, b, B, rng, alpha=0.05)` — iki grubu ayrı yeniden örnekleyerek.

In [ ]:
%%writefile extra_x3.py
import numpy as np


def paired_bootstrap(a, b, B, rng, alpha=0.05):
    ...


def unpaired_bootstrap(a, b, B, rng, alpha=0.05):
    ...

In [ ]:
# Kontrol (E3)
labcheck.run_file("extra_x3.py", CHECKS["x3"])

## Makale ödevi: Lugmayr ve ark. (2022): RePaint

[Lugmayr, Danelljan, Romero, Yu, Timofte, Van Gool — RePaint: Inpainting using Denoising Diffusion Probabilistic Models (CVPR 2022)](https://arxiv.org/abs/2201.09865)

d aşamasında eğittiğin koşulsuz (boş token'lı) U-Net'i yeniden eğitmeden tamamlama için kullan: bilinen bölgeyi her adımda ileri süreçle gürültülendirip yerleştir (Algoritma 1), atlama uzunluğu j ve yeniden örnekleme sayısı r için makale Tablo 4/Şekil 5 ablasyonunu yap. Maske türleri: yarım görüntü, kalın çizgi, rastgele %50 piksel. LPIPS yerine sınıflandırıcı öznitelik uzaklığı ve sınıf tutarlılığıyla değerlendir; x2'deki kâhin deneyindeki başarısızlık türünün gerçek modelde de görüldüğünü göster ve "reconstruction guidance" (bilinen bölgeye göre x̂₀ gradyanı) ile karşılaştır.


**Adımlar**

1. r = 1 (naif yerleştirme) ile r = 10'u aynı NFE bütçesinde karşılaştır.

**Kabul ölçütleri**

- Yeniden örnekleme, bilinen ve üretilen bölgeler arasındaki uyumu belirgin iyileştirir.
- Maske türüne göre sonuç tablosu ve örnek görseller.

## Makale ödevi: Isensee ve ark. (2021): nnU-Net

[Isensee, Jaeger, Kohl, Petersen, Maier-Hein — nnU-Net: a self-configuring method for deep learning-based biomedical image segmentation (Nature Methods 2021)](https://www.nature.com/articles/s41592-020-01008-z)

x1'deki segmentasyon hattını nnU-Net'in "sabit / kural tabanlı / deneysel" yapılandırma ayrımıyla yeniden düzenle: veri parmak izinden (görüntü boyutu, aralık, yoğunluk dağılımı) yama boyutu, batch boyutu ve ağ derinliğini kurallarla türet; Dice + CE, derin denetim, poly öğrenme oranı ve ağır veri artırma (sabit kurallar) uygula. Herkese açık küçük bir tıbbi veri setinde (ör. Medical Segmentation Decathlon'un bir görevi, 2B dilimler) 5 katlı çapraz doğrulamayla Dice raporla ve x3'teki eşleştirilmiş bootstrap ile tasarım seçimlerinin farkının anlamlılığını sına.


**Adımlar**

1. Ablasyon: derin denetim yok, yalnız CE, artırma yok.

**Kabul ölçütleri**

- Kural tabanlı yapılandırma elle ayarlanmış taban çizgisine yakın ya da üstün.
- Her ablasyon için güven aralıklı fark.

## Kaynaklar

- [Ho, Jain, Abbeel — DDPM (NeurIPS 2020) ve resmi U-Net mimarisi](https://arxiv.org/abs/2006.11239) — arXiv
- [Nichol & Dhariwal — Improved DDPM (ICML 2021)](https://arxiv.org/abs/2102.09672) — arXiv
- [Ho & Salimans — Classifier-Free Diffusion Guidance (2022)](https://arxiv.org/abs/2207.12598) — arXiv
- [Lugmayr ve ark. — RePaint: Inpainting using Denoising Diffusion Probabilistic Models (CVPR 2022)](https://arxiv.org/abs/2201.09865) — arXiv
- [Isensee ve ark. — nnU-Net: a self-configuring method for deep learning-based biomedical image segmentation (Nature Methods 2021)](https://www.nature.com/articles/s41592-020-01008-z) — ©
- [Milletari, Navab, Ahmadi — V-Net ve Dice kaybı (3DV 2016)](https://arxiv.org/abs/1606.04797) — arXiv
- [UvA DL Notebooks / fast.ai Part 2 — diffusion U-Net'leri](https://course.fast.ai/Lessons/part2.html) — Apache-2.0
- [Efron & Tibshirani — An Introduction to the Bootstrap](https://www.taylorfrancis.com/books/mono/10.1201/9780429246593/introduction-bootstrap-bradley-efron-tibshirani) — ©